# Pełny cykl życia modelu: Kedro + MLflow + JupyterLab

Zacznij od `warsztaty.md` w tym samym folderze: zawiera instrukcje, co kliknąć.
Notebook jest kompletnym przykładem z rozwiązaniami. Najpierw czytaj opis, potem kliknij
komórkę z kodem i naciśnij **Shift+Enter**. Liczba w `[1]` oznacza wykonanie;
`[*]` oznacza pracę. Poczekaj na liczbę, zanim przejdziesz dalej.

**Cel biznesowy:** wsadowa wycena 250 syntetycznych nieruchomości w PLN.
**Przed treningiem** ustalamy budżet RMSE ≤ 50 000 PLN i RMSE ≤ 70% błędu baseline.
To progi dydaktyczne, a nie wymagania rzeczywistego rynku. Kandydatów wybieramy po
walidacji, test sprawdzamy przy bramce wydania. Nie zmieniaj siatki po obejrzeniu testu.

Kedro wykonuje graf i obsługuje dane; MLflow zapisuje eksperymenty i wersje modelu;
JupyterLab jest miejscem eksploracji i sterowania tym samym kodem. Lokalny batch jest
naszym wdrożeniem. Nie uruchamiamy API modelu ani harmonogramu. Kedro-Viz jest dodatkiem.

Każde wykonanie od początku tworzy własny katalog i bazy w `.runtime/trening/pl/`.
Dane `data/` oraz istniejące notebooki pozostają osobnym demonstratorem.

## 0. Utwórz izolowane środowisko ćwiczenia

Ta komórka znajduje katalog projektu, tworzy katalog wykonania i ustawia połączenia
do lokalnych baz **zanim** wczytamy Kedro. Wypisany `WORKSPACE` zachowaj: tam znajdziesz
raporty i manifesty wydań. Wypisane polecenie MLflow wkleisz później do terminala.

In [ ]:
# ruff: noqa: E402
LANGUAGE = "pl"

import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists() and (path / "src/kedro_lasso").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from trening.lifecycle import (
    create_workspace,
    future_frame,
    monitor_inputs,
    quality_gate,
    validate_workspace_connections,
)

WORKSPACE = create_workspace(PROJECT_ROOT, LANGUAGE)
from kedro_lasso.commands import prepare_runtime, seed
from kedro_lasso.config import load_config

prepare_runtime()
config = load_config()
validate_workspace_connections(config, WORKSPACE)
seed()
tracking_uri = config["globals"]["tracking_uri"]
os.environ["MLFLOW_REGISTRY_URI"] = tracking_uri
policy = {"max_rmse_pln": 50_000, "max_baseline_ratio": 0.70}

print("Python:", sys.executable)
print("WORKSPACE:", WORKSPACE)
print("Release policy:", policy)
print("MLflow terminal command (run from the project root):")
print(
    f'uv run python -m mlflow server --backend-store-uri "{tracking_uri}" '
    f'--artifacts-destination "{WORKSPACE.as_posix()}/mlflow/artifacts" '
    "--host 127.0.0.1 --port 5001 --workers 1"
)

## 1. Połącz JupyterLab z Kedro

`%load_ext` ładuje rozszerzenie, a `%reload_kedro` odświeża projekt. Rozszerzenie
udostępnia `catalog`, `context`, `session` i `pipelines`. Po zmianie YAML lub kodu
użyj ponownie `%reload_kedro`. Do kolejnego wykonania twórz nową `KedroSession`.

In [ ]:
get_ipython().run_line_magic("load_ext", "kedro.ipython")
get_ipython().run_line_magic("reload_kedro", f'"{PROJECT_ROOT}"')
catalog = get_ipython().user_ns["catalog"]
context = get_ipython().user_ns["context"]
pipelines = get_ipython().user_ns["pipelines"]
print("Pipelines:", list(pipelines))
print("Catalog datasets:", list(catalog.keys()))
print("Source:", config["credentials"]["source_db"]["con"])
print("Warehouse:", config["credentials"]["warehouse_db"]["con"])

## 2. Poznaj i sprawdź dane

Połączenie SQLite i zapytanie SQL są w `conf/base/catalog.yml` i `credentials.yml`.
`catalog.load()` używa tej samej konfiguracji co zadanie wsadowe. `price_pln` to target
(wartość przewidywana), `property_id` to identyfikator, a pozostałe wskazane kolumny to
cechy. ID i target nie mogą być cechami. Braki cech uzupełni imputer wewnątrz modelu.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from kedro_lasso.domain import regression_metrics, validate_frame
from kedro_lasso.training import split_training

raw = catalog.load("raw_training")
raw_scoring = catalog.load("raw_scoring")
schema = context.params["data"]
features = schema["features"]
clean = validate_frame(raw, schema, training=True)
display(raw.head())
display(raw[features + [schema["target"]]].describe().T)
display(raw.isna().mean().rename("missing_rate").to_frame())
assert len(raw) == 1500 and len(raw_scoring) == 250

raw.plot.scatter(x="area_m2", y="price_pln", alpha=0.25, figsize=(8, 4))
plt.ylabel("Price [PLN]")
plt.title("Synthetic properties / Nieruchomości syntetyczne")
plt.show()

### Kontrolowany błąd: powtórzone ID

Poniższa komórka celowo tworzy błędne dane w pamięci. Oczekujemy komunikatu
`EXPECTED VALIDATION ERROR`, a nie zapisu do bazy. Obsługa wyjątku pozwala kontynuować.
**Zadanie A:** w nowej komórce usuń jedną cechę i sprawdź komunikat `Missing columns`.

In [ ]:
broken = pd.concat([raw, raw.iloc[:1]], ignore_index=True)
try:
    validate_frame(broken, schema, training=True)
except ValueError as error:
    print("EXPECTED VALIDATION ERROR:", error)
else:
    raise AssertionError("Duplicate IDs should have been rejected.")

### Rozłączne zbiory i architektura

Wywołujemy tę samą funkcję co node Kedro, zamiast kopiować logikę treningu do notebooka.
Train uczy parametry, validation wybiera `alpha`, test sprawdza gotowy wybór. Imputer
i scaler dopasowują się w modelu tylko na danych treningowych danego kandydata.
**Zadanie B:** kliknij dwukrotnie pliki `domain.py`, `training.py` i `pipeline_registry.py`
w przeglądarce plików. Znajdź definicję modelu, logowanie MLflow i składanie pipeline'ów.

In [ ]:
splits = split_training(clean, context.params["split"])
ids = {key: set(frame[schema["id_column"]]) for key, frame in splits.items()}
assert not (
    ids["train"] & ids["validation"]
    or ids["train"] & ids["test"]
    or ids["validation"] & ids["test"]
)
display(pd.Series({key: len(frame) for key, frame in splits.items()}, name="rows"))
print(pipelines["training"].describe())

## 3. Trenuj przez Kedro, śledź w MLflow

Uruchamiamy tylko `training`: walidacja → podział → eksperymenty → refit → raport.
Jeszcze nie publikujemy predykcji. `runtime_params` zmieniają nazwę eksperymentu i runu
bez edycji plików YAML. Zapisujemy ID sesji Kedro, które jest też wersją pliku modelu.
Run to jedno wykonanie zapisane w MLflow; podrzędne runy odpowiadają wartościom `alpha`.

In [ ]:
from kedro.framework.session import KedroSession
from kedro.io import Version
from kedro_datasets.pickle import PickleDataset

EXPERIMENT = "workshop-lifecycle"
REGISTERED_NAME = "workshop-housing-lasso"


def load_bundle(version):
    return PickleDataset(
        filepath=(WORKSPACE / "models" / "lasso.pkl").as_posix(),
        version=Version(version, None),
    ).load()


with KedroSession.create(
    project_path=PROJECT_ROOT,
    runtime_params={"tracking": {"experiment_name": EXPERIMENT, "run_name": "initial-sweep"}},
) as training_session:
    training_session.run(pipeline_name="training")
    initial_kedro_version = training_session.session_id

initial_bundle = load_bundle(initial_kedro_version)
initial_report = initial_bundle["report"]
display(pd.DataFrame(initial_report["trials"]).sort_values("rmse"))
display(
    pd.DataFrame(
        {
            "Lasso": initial_report["test_metrics"],
            "Baseline": initial_report["baseline_test_metrics"],
        }
    )
)
print("Kedro version:", initial_kedro_version)
print("MLflow run:", initial_report["run_id"])
print("Model URI:", initial_report["model_uri"])

### Obejrzyj eksperyment w interfejsie

Uruchom wypisane w sekcji 0 polecenie w terminalu. W nowej karcie otwórz
http://127.0.0.1:5001. Wybierz **Model training**, **Experiments** i `workshop-lifecycle`.
W zakładce **Runs** rozwiń strzałkę przy `initial-sweep`. Każde `alpha=...` ma
`validation_rmse`, natomiast test jest w runie nadrzędnym. Otwórz jego szczegóły:
metryki, parametry i artefakty `configuration.json`, `split_ids.json` oraz raport.
Nazwy szczegółowych zakładek mogą zależeć od szerokości okna; powiększ okno, gdy są ukryte.

**Zadanie C:** wskaż najlepsze `alpha` według validation RMSE i pokaż, że child runy
nie zawierają `test_rmse`. Nie wybieraj `alpha` na podstawie wyników testowych.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
client = MlflowClient(tracking_uri=tracking_uri, registry_uri=tracking_uri)
experiment_id = client.get_experiment_by_name(EXPERIMENT).experiment_id
children = client.search_runs(
    [experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{initial_report['run_id']}'",
)
assert len(children) == len(context.params["model"]["alphas"])
assert all("test_rmse" not in run.data.metrics for run in children)
display(pd.DataFrame([{"alpha": run.data.params["alpha"], **run.data.metrics} for run in children]))

trials = pd.DataFrame(initial_report["trials"]).sort_values("alpha")
trials.plot(x="alpha", y="rmse", marker="o", logx=True, figsize=(8, 4))
plt.ylabel("Validation RMSE [PLN]")
plt.show()

## 4. Bramka jakości, rejestr i manifest wydania

Dobry wynik runu nie wystarcza do wdrożenia. Sprawdzamy wcześniej ustalone progi.
Najpierw pokazujemy odrzucenie zmodyfikowanego raportu (bez treningu złego modelu).
Rejestr MLflow przechowuje wersje; alias `champion` jest zmiennym wskaźnikiem.
Manifest zapisuje konkretne wersje MLflow i Kedro oraz sumę kontrolną pliku.
Sam alias nie przełącza automatycznie katalogu Kedro.

In [ ]:
from copy import deepcopy

initial_gate = quality_gate(initial_report, policy)
display(initial_gate)
bad_report = deepcopy(initial_report)
bad_report["test_metrics"]["rmse"] = 200_000
assert not quality_gate(bad_report, policy)["passed"]
assert initial_gate["passed"], "Stop here: this candidate must not be released."

initial_registered = mlflow.register_model(initial_report["model_uri"], REGISTERED_NAME)
assert (
    client.get_model_version(REGISTERED_NAME, initial_registered.version).run_id
    == initial_report["run_id"]
)
client.set_model_version_tag(
    REGISTERED_NAME, initial_registered.version, "kedro_bundle_version", initial_kedro_version
)
client.set_model_version_tag(REGISTERED_NAME, initial_registered.version, "quality_gate", "passed")
client.set_registered_model_alias(REGISTERED_NAME, "champion", initial_registered.version)
print("Registry version:", initial_registered.version)
print("Alias:", client.get_model_version_by_alias(REGISTERED_NAME, "champion").version)

### Zapisz decyzję o wydaniu

`releases/initial.json` zawiera powiązania: commit kodu → lockfile → run MLflow →
wersja rejestru → wersja bundle Kedro. Zapisujemy go także jako artefakt runu.
Manifest jest lokalnym zapisem decyzji ćwiczeniowej; nie zastępuje uprawnień i procesu
akceptacji w zespole. Nie edytuj go ręcznie, aby obejść bramkę.

In [ ]:
import hashlib
import json
import subprocess
from datetime import UTC, datetime


def write_release(bundle, kedro_version, registry_version, gate, label):
    if not gate["passed"]:
        raise ValueError("Refusing to create a release for a rejected model.")
    report = bundle["report"]
    bundle_file = WORKSPACE / "models" / "lasso.pkl" / kedro_version / "lasso.pkl"
    commit = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True
    ).stdout.strip()
    manifest = {
        "created_at": datetime.now(UTC).isoformat(),
        "data_dir": WORKSPACE.as_posix(),
        "tracking_uri": tracking_uri,
        "run_id": report["run_id"],
        "model_uri": report["model_uri"],
        "registered_name": REGISTERED_NAME,
        "registry_version": str(registry_version),
        "kedro_bundle_version": kedro_version,
        "bundle_sha256": hashlib.sha256(bundle_file.read_bytes()).hexdigest(),
        "code_commit": commit,
        "uv_lock_sha256": hashlib.sha256((PROJECT_ROOT / "uv.lock").read_bytes()).hexdigest(),
        "data_fingerprints": report["fingerprints"],
        "quality_gate": gate,
    }
    directory = WORKSPACE / "releases"
    directory.mkdir(exist_ok=True)
    path = directory / f"{label}.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    client.log_artifact(report["run_id"], str(path), artifact_path="releases")
    return path


initial_release = write_release(
    initial_bundle, initial_kedro_version, initial_registered.version, initial_gate, "initial"
)
print("Initial release:", initial_release)
print(initial_release.read_text(encoding="utf-8"))

## 5. Wdróż jako batch poza notebookiem

Komórka uruchamia osobny proces Pythona: ten sam skrypt można wkleić do terminala
lub uruchamiać z zewnętrznego harmonogramu. Skrypt sprawdza powiązanie manifestu,
rejestru i bundle, a następnie wywołuje `session.run(pipeline_name="scoring",
load_versions={"model_bundle": wersja})`. Nie wybiera najnowszego modelu.

W warehouse zapisze się 250 predykcji w jednej transakcji. Liczba runów treningowych
ma pozostać taka sama. Kolorowe logi Kedro w terminalu są normalne.

In [ ]:
def run_release(path):
    command = [sys.executable, "-m", "trening.score_release", "--release", str(path)]
    print("Terminal command:", f'uv run python -m trening.score_release --release "{path}"')
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if result.returncode:
        raise RuntimeError(result.stdout + result.stderr)
    print(result.stdout[-1400:])


runs_before = len(client.search_runs([experiment_id]))
run_release(initial_release)
assert len(client.search_runs([experiment_id])) == runs_before

### Obejrzyj drugą bazę i sprawdź odtwarzalność

Wynik łączymy z `run_id`, nie tylko z nazwą modelu. Zapytanie pokazuje historię batchy.
Ładujemy też model bez Kedro z konkretnej wersji rejestru i porównujemy predykcje.
**Zadanie D:** ponów batch w terminalu. Liczba batchy wzrośnie, ale runów treningowych
nie przybędzie. Ponowne wykonanie pipeline'u nie jest idempotentne: tworzy nowy batch ID.

In [ ]:
from sqlalchemy import create_engine, text


def warehouse_query(sql):
    engine = create_engine(config["credentials"]["warehouse_db"]["con"])
    try:
        with engine.connect() as connection:
            return pd.read_sql(text(sql), connection)
    finally:
        engine.dispose()


history_sql = """SELECT b.batch_id, b.run_id, b.n_rows, b.created_at, m.best_alpha
FROM prediction_batches b JOIN model_runs m ON b.run_id = m.run_id
ORDER BY b.created_at, b.batch_id"""
display(warehouse_query(history_sql))
display(warehouse_query("SELECT * FROM predictions LIMIT 5"))
assert warehouse_query(history_sql).iloc[-1]["n_rows"] == 250

restored = mlflow.sklearn.load_model(f"models:/{REGISTERED_NAME}/{initial_registered.version}")
np.testing.assert_allclose(
    restored.predict(raw_scoring[features]), initial_bundle["model"].predict(raw_scoring[features])
)

## 6. Monitoruj wejścia, a jakość dopiero po uzyskaniu etykiet

Najpierw przesuwamy powierzchnię o 50 m² **tylko w kopii w pamięci**. Umowny alarm
to przesunięcie średniej o ponad 0,5 odchylenia standardowego danych referencyjnych
lub zmiana odsetka braków o ponad 0,10. To prosta heurystyka, nie test statystyczny.
Bez prawdziwych cen nie policzysz RMSE i nie udowodnisz pogorszenia modelu.

In [ ]:
shifted_inputs = raw_scoring.copy()
shifted_inputs["area_m2"] += 50
validate_frame(shifted_inputs, schema, training=False)
input_monitor = monitor_inputs(clean, shifted_inputs, features)
display(input_monitor)
assert bool(input_monitor.loc["area_m2", "alert"])

### Nadeszły ceny z nowego okresu

Generator projektu tworzy nowe rekordy z innym seedem i ID. Dodajemy 120 000 PLN do
targetu: symulujemy zmianę poziomu cen, której nie widać w samym rozkładzie cech.
To niezależne dane monitoringowe. Nie dopasowujemy do nich modelu. Monitoring zapisujemy
w osobnym eksperymencie MLflow, z powiązaniem do wdrożonego runu.
**Zadanie E:** porównaj alarm wejścia z alarmem jakości. Który wymaga etykiet?

In [ ]:
monitoring = future_frame(WORKSPACE, "monitoring", seed=84, id_offset=100_000)
monitoring = validate_frame(monitoring, schema, training=True)
initial_future_metrics = regression_metrics(
    monitoring[schema["target"]], restored.predict(monitoring[features])
)
display(initial_future_metrics)
assert initial_future_metrics["rmse"] > policy["max_rmse_pln"]

monitor_experiment = mlflow.set_experiment("workshop-monitoring")
with mlflow.start_run(
    experiment_id=monitor_experiment.experiment_id, run_name="labelled-future-period"
):
    mlflow.set_tags(
        {"model_run_id": initial_report["run_id"], "registry_version": initial_registered.version}
    )
    mlflow.log_metrics({f"observed_{key}": value for key, value in initial_future_metrics.items()})
    mlflow.log_dict(
        input_monitor.reset_index(names="feature").to_dict(orient="records"), "input_monitor.json"
    )
    mlflow.log_param("quality_alert", True)

## 7. Ponowny trening na nowym okresie

Używamy **innych** danych niż monitoring: inny seed i rozłączne ID. Zastępujemy tabelę
treningową i scoringową wyłącznie w własnej bazie ćwiczenia. Źródło v1 zachowujemy
w Parquet. Pipeline i konfiguracja modelu są te same. Zmieniamy run i opis zbioru.
Nie wykorzystujemy monitoringu ani końcowego nowego holdoutu do treningu.

In [ ]:
future_training = future_frame(WORKSPACE, "retraining", seed=73, id_offset=10_000, rows=1500)
future_scoring = future_frame(WORKSPACE, "scoring", seed=97, id_offset=30_000, rows=250).drop(
    columns=[schema["target"]]
)
assert not set(future_training.property_id) & set(monitoring.property_id)
raw.to_parquet(WORKSPACE / "source" / "training_v1.parquet", index=False)
raw_scoring.to_parquet(WORKSPACE / "source" / "scoring_v1.parquet", index=False)
source_engine = create_engine(config["credentials"]["source_db"]["con"])
try:
    with source_engine.begin() as connection:
        future_training.to_sql("training_properties", connection, if_exists="replace", index=False)
        future_scoring.to_sql("scoring_properties", connection, if_exists="replace", index=False)
finally:
    source_engine.dispose()

with KedroSession.create(
    project_path=PROJECT_ROOT,
    runtime_params={
        "tracking": {
            "experiment_name": EXPERIMENT,
            "run_name": "retrained-sweep",
            "dataset_name": "synthetic-future-v2",
        }
    },
) as retraining_session:
    retraining_session.run(pipeline_name="training")
    retrained_kedro_version = retraining_session.session_id

retrained_bundle = load_bundle(retrained_kedro_version)
retrained_report = retrained_bundle["report"]
assert retrained_report["run_id"] != initial_report["run_id"]
assert retrained_report["fingerprints"] != initial_report["fingerprints"]
display(
    pd.DataFrame(
        {
            "Initial test": initial_report["test_metrics"],
            "New-period test": retrained_report["test_metrics"],
        }
    )
)

### Challenger kontra champion na nowym, niezależnym holdoucie

Porównujemy oba modele na **tych samych nowych rekordach** z innego seeda i rozłącznymi
ID. Testy z poprzedniej tabeli nie są porównywalną oceną nowego okresu. Próg jakości
pozostaje ten sam; dodatkowo wymagamy spadku RMSE o co najmniej 50% względem starego
championa na tym holdoucie. To jednorazowa decyzja ćwiczeniowa; po dalszym dostrajaniu
potrzebny byłby kolejny zbiór końcowy.

In [ ]:
holdout = future_frame(WORKSPACE, "acceptance", seed=91, id_offset=20_000)
assert not set(holdout.property_id) & (
    set(future_training.property_id) | set(monitoring.property_id)
)
old_metrics = regression_metrics(
    holdout[schema["target"]], initial_bundle["model"].predict(holdout[features])
)
new_metrics = regression_metrics(
    holdout[schema["target"]], retrained_bundle["model"].predict(holdout[features])
)
display(pd.DataFrame({"Old champion": old_metrics, "Challenger": new_metrics}))
retrained_gate = quality_gate(retrained_report, policy)
independent_passed = (
    new_metrics["rmse"] <= policy["max_rmse_pln"]
    and new_metrics["rmse"] < old_metrics["rmse"] * 0.5
)
retrained_gate["checks"]["independent_period_passed"] = bool(independent_passed)
retrained_gate["independent_period"] = {
    "seed": 91,
    "old_metrics": old_metrics,
    "new_metrics": new_metrics,
}
retrained_gate["passed"] = bool(retrained_gate["passed"] and independent_passed)
assert retrained_gate["passed"], "Keep the current release; investigate this candidate."

retrained_registered = mlflow.register_model(retrained_report["model_uri"], REGISTERED_NAME)
client.set_model_version_tag(
    REGISTERED_NAME, retrained_registered.version, "kedro_bundle_version", retrained_kedro_version
)
client.set_model_version_tag(
    REGISTERED_NAME, retrained_registered.version, "quality_gate", "passed"
)
client.set_registered_model_alias(REGISTERED_NAME, "candidate", retrained_registered.version)
retrained_release = write_release(
    retrained_bundle,
    retrained_kedro_version,
    retrained_registered.version,
    retrained_gate,
    "retrained",
)
client.set_registered_model_alias(REGISTERED_NAME, "champion", retrained_registered.version)
client.set_registered_model_alias(REGISTERED_NAME, "previous", initial_registered.version)
run_release(retrained_release)
assert warehouse_query(history_sql).iloc[-1]["run_id"] == retrained_report["run_id"]

## 8. Rollback: przywróć konkretną wersję

Ćwiczymy awarię wdrożenia: przełączamy alias i uruchamiamy **stary manifest**.
Najnowszy plik modelu nadal jest v2, ale batch musi użyć v1. Historia wyników pozostaje.
W tym syntetycznym nowym reżimie v1 ma gorszą jakość; rollback jest tu ćwiczeniem
operacyjnym. Po sprawdzeniu wrócimy do zaakceptowanej v2.
**Zadanie F:** wyjaśnij, dlaczego samo przestawienie aliasu nie wystarczy skryptowi
z przypiętym manifestem i dlaczego rollback może pogorszyć jakość biznesową.

In [ ]:
client.set_registered_model_alias(REGISTERED_NAME, "champion", initial_registered.version)
runs_before_rollback = len(client.search_runs([experiment_id]))
run_release(initial_release)
rollback_batch = warehouse_query(history_sql).iloc[-1]
assert rollback_batch["run_id"] == initial_report["run_id"]
assert len(client.search_runs([experiment_id])) == runs_before_rollback
rollback_predictions = warehouse_query(
    "SELECT property_id, prediction FROM predictions "
    f"WHERE batch_id = '{rollback_batch['batch_id']}' ORDER BY property_id"
)
np.testing.assert_allclose(
    rollback_predictions.prediction,
    initial_bundle["model"].predict(future_scoring.sort_values("property_id")[features]),
)
print("Rollback verified:", rollback_batch["run_id"])
display(warehouse_query(history_sql))

## 9. Zakończ ćwiczenie i zachowaj dowody

Przywracamy zaakceptowaną v2. Zachowujemy v1 jako możliwość rollbacku; nie kasujemy
historii ani artefaktów. Politykę retencji i wycofania modelu uzgadnia się oddzielnie.
`summary.json` zawiera wynik ćwiczenia. Manifesty i raporty znajdziesz w `WORKSPACE`.

**Sprawdzenie końcowe:** potrafisz wskazać run modelu użytego do ostatniego batcha,
wyjaśnić różnicę między aliasem i wersją oraz wykonać scoring bez treningu.
W JupyterLab: **File → Save Notebook**. Terminal MLflow: **Ctrl+C**.
Przed ponownym wykonaniem od początku wybierz **Kernel → Restart Kernel and Clear
Outputs of All Cells…**. Nie kasuj baz, aby rozpocząć nowe ćwiczenie: nowy start
notebooka tworzy nowy katalog. Zobacz też instrukcję prowadzącego i zadanie końcowe.

In [ ]:
client.set_registered_model_alias(REGISTERED_NAME, "champion", retrained_registered.version)
run_release(retrained_release)
client.set_model_version_tag(
    REGISTERED_NAME, initial_registered.version, "deployment_status", "rollback_available"
)
client.set_model_version_tag(
    REGISTERED_NAME, retrained_registered.version, "deployment_status", "active"
)
history = warehouse_query(history_sql)
assert history.iloc[-1]["run_id"] == retrained_report["run_id"]
assert str(client.get_model_version_by_alias(REGISTERED_NAME, "champion").version) == str(
    retrained_registered.version
)
assert history.n_rows.eq(250).all()
assert set(history.run_id) == {initial_report["run_id"], retrained_report["run_id"]}
summary = {
    "language": LANGUAGE,
    "workspace": str(WORKSPACE),
    "initial_run_id": initial_report["run_id"],
    "active_run_id": retrained_report["run_id"],
    "batch_count": len(history),
    "rollback_verified": True,
    "retraining_improved_rmse": bool(independent_passed),
    "initial_release": str(initial_release),
    "active_release": str(retrained_release),
}
(WORKSPACE / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(history)
display(summary)

## 10. Gdzie kończy się demonstracja

Zrealizowaliśmy: cel i kryteria → dane i kontrakt → EDA → walidacja → eksperymenty →
artefakt → rejestr → decyzja wydania → batch w innej bazie → monitoring → retraining →
rollback → zapis dowodów. Monitoring i promocja są uruchamiane ręcznie.
Współdzielone środowisko potrzebowałoby osobnych uprawnień, PostgreSQL, magazynu
obiektowego, migracji, retencji, orkiestracji, rzeczywistych etykiet i walidacji czasowej.
Identyfikatory i seedy w symulacji dowodzą rozłączności ćwiczenia, nie realnej niezależności
czasowej rynku. Nie używaj generatora syntetycznego do oceny jakości biznesowej.

Źródła: [Kedro i Jupyter](https://docs.kedro.org/en/stable/integrations-and-plugins/notebooks_and_ipython/kedro_and_notebooks/),
[MLflow Model Registry](https://mlflow.org/docs/latest/ml/model-registry/workflow/).
Kod treningu znajduje się w `src/kedro_lasso/training.py`; narzędzia dydaktyczne w `trening/`.